# 01 — Secure Software Development Lifecycle (SSDLC)

## What This Is
The Secure SDLC integrates security activities into every phase of software development — requirements, design, implementation, testing, and deployment. It shifts security left (find bugs earlier, when they're cheap to fix) rather than bolting it on at the end.

## Why It Matters
The cost of fixing a security bug scales dramatically with discovery phase: requirements ($100) → design ($500) → implementation ($1,000) → testing ($5,000) → production ($100,000+). Microsoft's SDL (2004) reduced security vulnerabilities by 50-60% across products. NIST SP 800-218 (SSDF) defines the federal SDLC framework.

## Where the Community Stands
DevSecOps integrates SSDLC into CI/CD pipelines. GitHub Advanced Security, GitLab SAST, and Snyk provide automated security scanning in pull requests. OWASP SAMM (Software Assurance Maturity Model) provides a measurement framework. Supply chain security (SLSA) has become a top priority since SolarWinds.

## SSDLC Phases
- **Requirements**: security requirements, threat modelling, abuse cases
- **Design**: architecture review, attack surface analysis, security patterns
- **Implementation**: secure coding standards, SAST, dependency checking
- **Testing**: DAST, penetration testing, security regression tests
- **Deployment**: hardened configuration, SBOM, runtime monitoring

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# OWASP SAMM Business Functions and Practices
SAMM_FRAMEWORK = {
    'Governance': [
        'Strategy & Metrics',
        'Policy & Compliance',
        'Education & Guidance',
    ],
    'Design': [
        'Threat Assessment',
        'Security Requirements',
        'Security Architecture',
    ],
    'Implementation': [
        'Secure Build',
        'Secure Deployment',
        'Defect Management',
    ],
    'Verification': [
        'Architecture Assessment',
        'Requirements-driven Testing',
        'Security Testing',
    ],
    'Operations': [
        'Incident Management',
        'Environment Management',
        'Operational Management',
    ],
}

SAMM_MATURITY_LEVELS = {
    0: 'Not implemented',
    1: 'Ad-hoc activities — partially defined',
    2: 'Standardised — consistently applied',
    3: 'Optimised — measured and continuously improved',
}

for function, practices in SAMM_FRAMEWORK.items():
    print(f'{function}:')
    for p in practices:
        print(f'  - {p}')

In [ ]:
@dataclass
class SAMMAssessment:
    team: str
    scores: Dict[str, Dict[str, int]] = field(default_factory=dict)

    def score(self, function: str, practice: str, level: int) -> None:
        self.scores.setdefault(function, {})[practice] = level

    def function_score(self, function: str) -> float:
        practices = self.scores.get(function, {})
        if not practices:
            return 0.0
        return sum(practices.values()) / (len(practices) * 3)  # max=3

    def overall_score(self) -> float:
        return sum(self.function_score(f) for f in SAMM_FRAMEWORK) / len(SAMM_FRAMEWORK)

    def report(self) -> None:
        print(f'OWASP SAMM Assessment: {self.team}')
        print(f'Overall maturity: {self.overall_score():.2f}/1.00')
        print()
        for function in SAMM_FRAMEWORK:
            fs = self.function_score(function)
            bar = '#' * int(fs * 20)
            print(f'  {function:<18} {fs:.2f}  {bar}')
            for practice, level in self.scores.get(function, {}).items():
                print(f'    {practice:<30} Level {level}: {SAMM_MATURITY_LEVELS[level]}')

# Example startup team assessment
assessment = SAMMAssessment('Startup Engineering')
assessment.score('Governance',     'Strategy & Metrics',          1)
assessment.score('Governance',     'Policy & Compliance',          0)
assessment.score('Governance',     'Education & Guidance',         1)
assessment.score('Design',         'Threat Assessment',            1)
assessment.score('Design',         'Security Requirements',        0)
assessment.score('Design',         'Security Architecture',        1)
assessment.score('Implementation', 'Secure Build',                 2)
assessment.score('Implementation', 'Secure Deployment',            1)
assessment.score('Implementation', 'Defect Management',            1)
assessment.score('Verification',   'Architecture Assessment',      0)
assessment.score('Verification',   'Requirements-driven Testing',  0)
assessment.score('Verification',   'Security Testing',             1)
assessment.score('Operations',     'Incident Management',          1)
assessment.score('Operations',     'Environment Management',       1)
assessment.score('Operations',     'Operational Management',       0)
assessment.report()

## Security Gates in CI/CD
Security gates automatically block deployments that fail security checks: SAST findings above threshold, known CVEs in dependencies, failed DAST scans, or missing security tests coverage.

In [ ]:
@dataclass
class SecurityGateResult:
    gate_name:  str
    passed:     bool
    severity:   str   # BLOCKING/WARNING/INFO
    detail:     str

class CICDSecurityGates:
    def __init__(self):
        self.results: List[SecurityGateResult] = []

    def check_sast(self, critical_findings: int, high_findings: int) -> SecurityGateResult:
        passed = critical_findings == 0 and high_findings <= 3
        r = SecurityGateResult(
            'SAST Scan', passed, 'BLOCKING' if not passed else 'INFO',
            f'Critical={critical_findings} High={high_findings}'
        )
        self.results.append(r)
        return r

    def check_sca(self, critical_cves: List[str], high_cves: List[str]) -> SecurityGateResult:
        passed = len(critical_cves) == 0
        cve_str = ', '.join((critical_cves + high_cves)[:5])
        r = SecurityGateResult(
            'SCA/Dependencies', passed, 'BLOCKING' if not passed else 'WARNING',
            f'Critical CVEs: {cve_str or "none"} | High: {len(high_cves)}'
        )
        self.results.append(r)
        return r

    def check_secrets(self, secrets_found: int) -> SecurityGateResult:
        passed = secrets_found == 0
        r = SecurityGateResult(
            'Secret Detection', passed, 'BLOCKING' if not passed else 'INFO',
            f'{secrets_found} secrets detected in code/config'
        )
        self.results.append(r)
        return r

    def check_sbom(self, sbom_present: bool, signed: bool) -> SecurityGateResult:
        passed = sbom_present and signed
        r = SecurityGateResult(
            'SBOM Attestation', passed, 'WARNING',
            f'SBOM present={sbom_present} signed={signed}'
        )
        self.results.append(r)
        return r

    def deploy_decision(self) -> bool:
        blocking_fails = [r for r in self.results
                          if r.severity == 'BLOCKING' and not r.passed]
        return len(blocking_fails) == 0

gates = CICDSecurityGates()
gates.check_sast(critical_findings=0, high_findings=2)
gates.check_sca(critical_cves=['CVE-2023-44487'], high_cves=['CVE-2023-5363'])
gates.check_secrets(secrets_found=0)
gates.check_sbom(sbom_present=True, signed=True)

print('CI/CD Security Gate Results:')
for r in gates.results:
    status = 'PASS' if r.passed else 'FAIL'
    print(f'  [{r.severity}] {r.gate_name:<25} {status}  {r.detail}')

decision = gates.deploy_decision()
print(f'\nDeploy decision: {"BLOCKED" if not decision else "PROCEED"}')